In [1]:
import numpy as np
import pandas as pd
import h5py

In [2]:
valid = pd.read_csv('../../../../results/translation_efficiency/training_data/2_bin/valid.tsv', sep='\t')
scores = pd.read_csv('../../../../results/translation_efficiency/training_data/2_bin/models/pcv2-l48-d1536/valid_predictions.tsv', sep='\t')
valid['scores'] = scores['probability_positive']

In [3]:
valid.head()

,gene,chr,strand,TPM,seq,label,scores
0,AT1G01030,1,-,1.088918,ATATACATATATAGTTTTCTTCCGATTCTAGGGTTTTCATATTTCC...,1,0.496288
1,AT1G01070,1,-,5.516785,ATAAAAAATTAAAGGGTGTAGTATGAAGTGCTTGATTAACTTTTCA...,1,0.556555
2,AT1G01100,1,-,323.478734,TAAATATGTCAAATGAAGAATTATGTTACTAGTGCGAAAAGGACAT...,1,0.903292
3,AT1G01120,1,-,95.581248,ATTAACAAGTTATGTTGTTTATCAAAAACTTCAAAAAAAAAAAAAA...,1,0.783045
4,AT1G01200,1,-,2.641557,TGGAGCATAAGAGCTTTTTTAGGTGGTCTCCAGACAAATCGTGGGT...,1,0.693615


In [4]:
ism = pd.read_csv('../../../../results/translation_efficiency/training_data/2_bin/valid_ism.tsv', sep='\t')
scores = pd.read_csv('../../../../results/translation_efficiency/training_data/2_bin/models/pcv2-l48-d1536/valid_ism_scores.tsv', sep='\t')
ism['scores'] = scores['probability_positive']

In [5]:
ism.head()

,gene,original_index,pos,original_base,mutated_base,seq,scores
0,AT1G01030,0,-100,T,A,ATATACATATATAGTTTTCTTCCGATTCTAGGGTTTTCATATTTCC...,0.495933
1,AT1G01030,0,-100,T,C,ATATACATATATAGTTTTCTTCCGATTCTAGGGTTTTCATATTTCC...,0.504070
2,AT1G01030,0,-100,T,G,ATATACATATATAGTTTTCTTCCGATTCTAGGGTTTTCATATTTCC...,0.498618
3,AT1G01030,0,-99,T,A,ATATACATATATAGTTTTCTTCCGATTCTAGGGTTTTCATATTTCC...,0.494417
4,AT1G01030,0,-99,T,C,ATATACATATATAGTTTTCTTCCGATTCTAGGGTTTTCATATTTCC...,0.504719


In [6]:
num_seqs = valid.shape[0]
seq_length = 500
bases = ['A', 'C', 'G', 'T']

# One-hot encoding function
def one_hot_encode(seq):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
    return np.array([mapping.get(base, [0,0,0,0]) for base in seq])

one_hot_seqs = np.array([one_hot_encode(seq) for seq in valid['seq']])

In [7]:
one_hot_seqs.shape

(5492, 500, 4)

In [8]:
importance_scores = np.zeros((num_seqs, seq_length, 4))

for idx, group in ism.groupby('original_index'):
    original_score = valid.loc[idx, 'scores']
    seq = valid.loc[idx, 'seq']
    for _, row in group.iterrows():
        pos = 500 + row['pos']  # original pos is negative relative to ATG
        mut_base = row['mutated_base']
        base_idx = bases.index(mut_base)
        
        original_logodds = np.log2(original_score / (1 - original_score + 1e-8))
        mutated_logodds = np.log2(row['scores'] / (1 - row['scores'] + 1e-8))
        
        delta_logodds = mutated_logodds - original_logodds  # Signed log-odds difference
        abs_delta_logodds = abs(delta_logodds)  # Absolute log-odds difference
        importance_scores[idx, pos, base_idx] = abs_delta_logodds

In [9]:
one_hot_seqs = np.transpose(one_hot_seqs, (0, 2, 1))
one_hot_seqs.shape

(5492, 4, 500)

In [10]:
np.savez_compressed('ohe.npz', one_hot_seqs)

In [11]:
importance_scores.shape

(5492, 500, 4)

In [12]:
importance_scores = np.transpose(importance_scores, (0, 2, 1))
np.savez_compressed('shap.npz', importance_scores)

In [13]:
import numpy as np

data = np.load('shap.npz')['arr_0']
print(data.shape)
print("max shap:", np.max(data))
print("min shap:", np.min(data))
print("mean shap:", np.mean(data))

(5492, 4, 500)
max shap: 2.646835935323296
min shap: 0.0
mean shap: 0.00866883403086174


In [16]:
import numpy as np

ohe = np.load('ohe.npz')['arr_0']      # shape (N, 4, 500)
shap = np.load('shap.npz')['arr_0']    # shape (N, 4, 500)

ohe_roi  = ohe[:, :, -100:]
shap_roi = shap[:, :, -100:]

print(ohe_roi.shape, shap_roi.shape)  # 都应是 (N, 4, 100)

np.savez('ohe_roi.npz', ohe_roi)
np.savez('shap_roi.npz', shap_roi)

(5492, 4, 100) (5492, 4, 100)


In [17]:
import modiscolite
import numpy as np

In [18]:
resOneHot = np.load('ohe_roi.npz')
resOneHot = resOneHot['arr_0']
resOneHot.shape

(5492, 4, 100)

In [19]:
resOneHot = np.transpose(resOneHot, (0, 2, 1))
resOneHot.shape

(5492, 100, 4)

In [20]:
loaded = np.load('shap_roi.npz')

In [21]:
resProb_array = loaded['arr_0']
resProb_array = np.transpose(resProb_array, (0, 2, 1))
resProb_array.shape

(5492, 100, 4)

In [22]:
mask = (resProb_array == 0)
noise = np.random.normal(loc=0.0, scale=1e-6, size=resProb_array.shape)
shap_jitter = np.where(mask, noise, resProb_array)

In [23]:
pos_patterns, neg_patterns = modiscolite.tfmodisco.TFMoDISco(
    hypothetical_contribs=shap_jitter,
    one_hot=resOneHot,
    max_seqlets_per_metacluster=200_000,
    sliding_window_size=10,
    flank_size=5,
    verbose=True)

Using 2271 positive seqlets
Extracted 2367 negative seqlets


In [24]:
modiscolite.io.save_hdf5('modisco_results.h5', pos_patterns, neg_patterns, window_size = 10)